# 08.5 - BERT-style Models

**Phase:** 08 - Transformers

**Status:** VERIFIED

---

## 1. What Are We Solving?

BERT is an **encoder-only** transformer pre-trained with **Masked Language Modeling (MLM)**: it reads the whole sentence at once and fills in blanks. The result is a model that produces rich contextual representations for *understanding* tasks - classification, NER, QA, similarity.

## 2. Why Does This Matter?

BERT proved that pre-training an encoder on massive text, then fine-tuning on a small labeled dataset, works extremely well. Understanding the MLM objective and the bidirectional attention behind it explains a large fraction of modern NLP.

## 3. Prerequisites

- Transformer block (08.1), self-attention (08.2), positional encoding (08.3)
- Encoder-decoder architecture (08.4)

## 4. Learning Objectives

By the end of this unit, you should:
- Build an encoder-only transformer with learned embeddings
- Pre-train with Masked Language Modeling on a synthetic corpus with a hidden rule
- Show bidirectional attention beats causal attention when the answer needs right context
- Fine-tune a [CLS]-head on a downstream classification task

## 5. Mental Model

BERT is a reading-comprehension specialist. It reads the whole sentence at once (**bidirectional**, no causal mask) and fills in blanks to learn language. For downstream tasks, we read off the [CLS] token or the token outputs.

```text
Pretraining:  'the cat [MASK] on the mat' -> predict 'sat' using BOTH sides
Fine-tuning:  '[CLS] this movie is great [SEP]' -> positive    (CLS head)
```


## 6. Setup


In [1]:
import matplotlib
matplotlib.use('Agg')
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)
np.random.seed(42)
print('torch', torch.__version__)


torch 2.13.0+cpu


## 7. Attention Block + Tiny BERT Encoder

Same attention as 08.4, but now the encoder can optionally apply a **causal mask** so we can compare bidirectional vs causal behavior.


In [2]:
def causal_mask(n):
    return torch.tril(torch.ones(n, n, dtype=torch.bool))

class MultiHeadAttention(nn.Module):
    def __init__(self, d, h):
        super().__init__()
        self.h, self.dk = h, d // h
        self.wq, self.wk, self.wv, self.wo = (nn.Linear(d, d) for _ in range(4))
    def forward(self, x, mask=None):
        B, T, _ = x.shape
        q = self.wq(x).view(B, T, self.h, self.dk).transpose(1, 2)
        k = self.wk(x).view(B, T, self.h, self.dk).transpose(1, 2)
        v = self.wv(x).view(B, T, self.h, self.dk).transpose(1, 2)
        s = q @ k.transpose(-2, -1) / math.sqrt(self.dk)
        if mask is not None:
            s = s.masked_fill(~mask[None, None], float('-inf'))
        w = F.softmax(s, dim=-1)
        o = (w @ v).transpose(1, 2).reshape(B, T, -1)
        return self.wo(o), w

class EncoderBlock(nn.Module):
    def __init__(self, d, h):
        super().__init__()
        self.attn = MultiHeadAttention(d, h)
        self.n1 = nn.LayerNorm(d)
        self.ff = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d))
        self.n2 = nn.LayerNorm(d)
    def forward(self, x, mask=None):
        a, _ = self.attn(x, mask=mask)
        x = self.n1(x + a)
        return self.n2(x + self.ff(x))

class TinyBERT(nn.Module):
    def __init__(self, vocab, d=24, h=4, n_layers=2, max_len=32):
        super().__init__()
        self.emb = nn.Embedding(vocab, d)
        self.pos = nn.Parameter(torch.randn(1, max_len, d) * 0.02)
        self.blocks = nn.ModuleList([EncoderBlock(d, h) for _ in range(n_layers)])
    def forward(self, x, mask=None):
        h = self.emb(x) + self.pos[:, :x.size(1)]
        for b in self.blocks:
            h = b(h, mask=mask)
        return h

bert = TinyBERT(vocab=12)
h = bert(torch.randint(0, 12, (2, 8)))
print('TinyBERT output:', tuple(h.shape))
n = sum(p.numel() for p in bert.parameters())
print('TinyBERT parameters:', f'{n:,}')


TinyBERT output: (2, 8, 24)
TinyBERT parameters: 15,504


## 8. A Synthetic Corpus with a Hidden Rule

Vocab = 10 content tokens forming pairs **e, e+1** (even, odd). Corpus sentences are built from 4 random pairs. The rule that lets MLM fill blanks: **an odd token's left neighbor is its even partner; an even token's right neighbor is its odd partner.**


In [3]:
VPAIRS, L = 5, 8          # 5 pairs (tokens 0..9), sequences of length 8
MASK, CLS = 10, 11         # special tokens
EVENS = torch.arange(0, 10, 2)

def gen_corpus(n):
    evens = EVENS[torch.randint(len(EVENS), (n, L // 2))]
    seqs = torch.stack([evens, evens + 1], dim=-1).reshape(n, L)
    return seqs

def mask_tokens(seqs, p=0.15):
    seqs = seqs.clone()
    targets = torch.full_like(seqs, -100)
    m = torch.rand_like(seqs.float()) < p
    targets[m] = seqs[m]
    seqs[m] = MASK
    return seqs, targets

s = gen_corpus(3)
sm, tm = mask_tokens(s)
for i in range(3):
    print(f'example {i}: {sm[i].tolist()}  (masked targets: {tm[i].tolist()})')
print('\nRule: token 2k is always immediately followed by token 2k+1.')


example 0: [10, 1, 8, 9, 10, 3, 10, 3]  (masked targets: [0, -100, -100, -100, 2, -100, 2, -100])
example 1: [0, 1, 8, 9, 0, 1, 4, 5]  (masked targets: [-100, -100, -100, -100, -100, -100, -100, -100])
example 2: [8, 9, 0, 1, 10, 10, 6, 7]  (masked targets: [-100, -100, -100, -100, 6, 7, -100, -100])

Rule: token 2k is always immediately followed by token 2k+1.


## 9. Pretrain with Masked Language Modeling

15% of positions are replaced with [MASK]; the head must predict the original token from **both** left and right context.


In [4]:
def train_mlm(causal=False, steps=300, seed=7):
    torch.manual_seed(seed)
    model = TinyBERT(vocab=12)
    head = nn.Linear(24, 12)
    opt = torch.optim.Adam(list(model.parameters()) + list(head.parameters()), lr=3e-3)
    loss_fn = nn.CrossEntropyLoss()
    for _ in range(steps):
        sm, tm = mask_tokens(gen_corpus(64))
        cm = causal_mask(L) if causal else None
        opt.zero_grad()
        logits = head(model(sm, mask=cm)).reshape(-1, 12)
        loss = loss_fn(logits, tm.reshape(-1))
        loss.backward()
        opt.step()
    return model, head

bi_model, bi_head = train_mlm(causal=False, steps=300)
sm_e, tm_e = mask_tokens(gen_corpus(64))
end_loss = nn.CrossEntropyLoss()(bi_head(bi_model(sm_e)).reshape(-1, 12), tm_e.reshape(-1)).item()
print('\nBidirectional MLM pretraining complete (loss on a held-out masked batch: %.3f).' % end_loss)



Bidirectional MLM pretraining complete (loss on a held-out masked batch: 0.497).


## 10. Why Bidirectional? The Right-Context Experiment

An **even** mask position can only be recovered from its *right* neighbor (the odd partner). We evaluate the way BERT is actually probed: mask **one** position at a time. The causal model must guess from nothing - and its only option is the uniform prior over evens (1 in 5 = 20%).


In [5]:
ca_model, ca_head = train_mlm(causal=True, steps=300, seed=11)

def acc_at_pos(model, head, causal, pos):
    seqs = gen_corpus(640)
    targets = torch.full_like(seqs, -100)
    targets[:, pos] = seqs[:, pos]
    seqs = seqs.clone(); seqs[:, pos] = MASK
    cm = causal_mask(L) if causal else None
    with torch.no_grad():
        preds = head(model(seqs, mask=cm)).argmax(-1)
    return (preds[:, pos] == targets[:, pos]).float().mean().item()

for pos in [0, 2, 4, 6]:
    bi = acc_at_pos(bi_model, bi_head, False, pos)
    ca = acc_at_pos(ca_model, ca_head, True, pos)
    print(f'even pos {pos}: bidirectional {bi*100:3.0f}%   causal {ca*100:3.0f}%')
print('\nOnly the RIGHT neighbor reveals an even token - bidirectional attention sees it,')
print('the causal model can only fall back on the 20% prior over evens.')


even pos 0: bidirectional 100%   causal  21%


even pos 2: bidirectional 100%   causal  21%


even pos 4: bidirectional 100%   causal  18%


even pos 6: bidirectional 100%   causal  19%

Only the RIGHT neighbor reveals an even token - bidirectional attention sees it,
the causal model can only fall back on the 20% prior over evens.


## 11. Fill in the Blank (like a tiny BERT)


In [6]:
example = torch.tensor([[MASK, 3, 6, 7, 8, 9]])   # pos0: even, hidden; right neighbor = 3 -> 2
with torch.no_grad():
    logits = bi_head(bi_model(example))
    probs = F.softmax(logits[0, 0], dim=-1)
top = probs.topk(3)
print('Predictions for masked position (true answer 2):')
for v, i in zip(top.values, top.indices):
    print(f'  token {i.item():2d}: {v.item()*100:5.1f}%')
print(f'P(true token=2): {probs[2].item()*100:.1f}%')


Predictions for masked position (true answer 2):
  token  2:  99.0%
  token  6:   0.4%
  token  8:   0.2%
P(true token=2): 99.0%


## 12. Challenge: The Odd Positions Need Only LEFT Context

Probe the **odd** positions. Each odd token is given away by its LEFT partner, which causal attention can use too - so both models should now solve it.


In [7]:
for pos in [1, 3, 5, 7]:
    bi = acc_at_pos(bi_model, bi_head, False, pos)
    ca = acc_at_pos(ca_model, ca_head, True, pos)
    print(f'odd pos {pos}:  bidirectional {bi*100:3.0f}%   causal {ca*100:3.0f}%')
print('\nOdd tokens are revealed by the token directly on their left -')
print('both attention directions can use that information.')


odd pos 1:  bidirectional 100%   causal 100%


odd pos 3:  bidirectional 100%   causal 100%


odd pos 5:  bidirectional 100%   causal 100%


odd pos 7:  bidirectional 100%   causal 100%

Odd tokens are revealed by the token directly on their left -
both attention directions can use that information.


## 13. Downstream Task: Fine-tune the [CLS] Head

The classic BERT move: keep the pre-trained encoder, attach a small head to the [CLS] token, and fine-tune on a labeled task. Our senti-class is artificial but the machinery is real.


In [8]:
# label = 1 if the sentence contains a pair from {even 4,6,8} (positive words)
def gen_labels(n):
    seqs = gen_corpus(n)
    pos = (seqs % 2 == 0) & (seqs >= 4)
    y = pos.any(dim=1).long()
    return seqs, y

cls_head = nn.Linear(24, 2)
opt = torch.optim.Adam(list(bi_model.parameters()) + list(cls_head.parameters()), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

for step in range(1, 251):
    seqs, y = gen_labels(64)
    x = torch.cat([torch.full((64, 1), CLS), seqs], dim=1)
    opt.zero_grad()
    h = bi_model(x)
    loss = loss_fn(cls_head(h[:, 0]), y)
    loss.backward()
    opt.step()
    if step % 100 == 0:
        with torch.no_grad():
            acc = (cls_head(bi_model(x))[:, 0].argmax(-1) == y).float().mean().item()
        print(f'step {step:3d}: loss={loss.item():.3f} train_acc={acc*100:.0f}%')

seqs_t, y_t = gen_labels(500)
x_t = torch.cat([torch.full((500, 1), CLS), seqs_t], dim=1)
with torch.no_grad():
    acc = (cls_head(bi_model(x_t))[:, 0].argmax(-1) == y_t).float().mean().item()
print(f'\nFine-tuned [CLS] classification accuracy: {acc*100:.0f}%')


step 100: loss=0.005 train_acc=100%


step 200: loss=0.002 train_acc=100%



Fine-tuned [CLS] classification accuracy: 100%


## 14. Debugging

| Symptom | Possible Cause | Verify | Fix |
|---|---|---|---|
| [CLS] output is poor | Head not attached to position 0 | Check head input | Use h[:, 0] |
| MLM loss stuck | Masked targets include padding | Check ignore_index=-100 | Mask loss on targets |
| Fine-tune overfits quickly | LR too high on tiny data | Compare train/val loss | Warmup or lower LR |
| Tokens ignored | Attention mask not respected | Inspect masked attention | Use length masks |

## 15. Real-World Considerations

- Search engines rank with BERT-style embeddings; NER uses token outputs; QA reads spans.
- BERT max length 512; long docs need truncation or Longformer.
- RoBERTa/ALBERT/DistilBERT are the same architecture with better pre-training or smaller size.

## 16. Common Mistakes

- Using BERT for generation (it cannot; use a decoder-only model).
- Fine-tuning at full pretraining LR (destroys representations).
- Ignoring the attention mask for padded batches.

## 17. When NOT to Use

- Generation and open-ended tasks -> GPT-style decoder.
- Multilingual corpora without a multilingual tokenizer.
- Very long documents (512-token budget) without a long-context variant.

## 18. Closed-Book Recall

1. Why is BERT bidirectional but GPT is not?
2. What does the [CLS] token represent and where is its head attached?
3. How does MLM pre-training help downstream tasks?
4. When would you pick BERT over a decoder-only model?

## 19. Teach-Back Questions

Explain to another person:

- The fill-in-the-blank pre-training objective.
- The right-context experiment that proved bidirectional matters.
- How [CLS] head fine-tuning differs from full fine-tuning.

## 20. Summary

You built an encoder-only transformer, pre-trained it with MLM on a synthetic corpus with a hidden pair rule, proved bidirectional attention wins when answers need right context, filled in a blank, and fine-tuned a [CLS] head for classification.

## 21. Further Experiment

- Toast a REAL text corpus (a few thousand sentences) through a char BPE and pre-train TinyBERT on it.
- Compare RoBERTa-style masking (not piecewise) vs the classic 15%-masking.

## 22. Verification Status

```
STATUS: VERIFIED
EXECUTION: PASS
DEPENDENCIES: numpy, torch
OUTPUTS: PASS
LAST VERIFIED: 2026-08-29
```
